<a href="https://colab.research.google.com/github/it25101136/IT2011-Stress-Level-Prediction-/blob/Chanul/6_Weerasekara_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Shared Setup: Load Data and Split
from google.colab import drive
import pandas as pd
from sklearn.model_selection import train_test_split
import os

drive.mount('/content/drive', force_remount=True)

df_master = pd.read_csv('/content/drive/MyDrive/IT2011_G12_AI_Project/results/outputs/FINAL_ML_READY_DATA.csv')

# FIX: Prevent Data Leakage
# Initially, the model hit 1.0 (100%) accuracy because it was "cheating".
# 'stress_score' is the direct numerical equivalent of the target 'stress_level'.
# Keeping it allowed the AI to read the answer key instead of learning patterns.
# By dropping it, we force the AI to predict stress using actual student habits
# (sleep, screen time, etc.). The accuracy drops to a realistic 0.69, proving
# the model is genuinely working and not suffering from data leakage.

# FIX: Drop BOTH the target category and the direct score to prevent data leakage
X = df_master.drop(columns=['stress_level', 'stress_score'])
y = df_master['stress_level']

# 80% Training, 20% Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Mounted at /content/drive
Training data shape: (2400, 7)
Testing data shape: (600, 7)


In [2]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

print("Executing Member 6 Model: MLP Neural Network")

param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'activation': ['relu', 'tanh'],
    'alpha': [0.0001, 0.05]
}

mlp = MLPClassifier(max_iter=2000, random_state=42)
grid_search = GridSearchCV(mlp, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Optimum Parameters found: {grid_search.best_params_}")
print(f"Validation Accuracy: {accuracy:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

pd.DataFrame({'Model': ['MLP Neural Network'], 'Accuracy': [accuracy]}).to_csv(
    '/content/drive/MyDrive/IT2011_G12_AI_Project/results/outputs/Score_Weerasekara.csv', index=False)

Executing Member 6 Model: MLP Neural Network
Optimum Parameters found: {'activation': 'tanh', 'alpha': 0.0001, 'hidden_layer_sizes': (50, 50)}
Validation Accuracy: 0.6967

Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.29      0.40        62
           1       0.73      0.73      0.73       239
           2       0.67      0.76      0.71       299

    accuracy                           0.70       600
   macro avg       0.69      0.59      0.62       600
weighted avg       0.70      0.70      0.69       600

